# SAT-14A · Blinded audit, step 1: build the review sheets  (CPU, ~5 min)

**Attach:** `sat-ext` + `sat-splits` + `satelite-coco` + `sat-results-e` (the grouped YOLO11s seed-42 run **with its saved
detections for `standard` and `sahi320`**). **Accelerator:** None. **Internet:** off. Do **not** attach the notebook-13
relabel outputs (the audit must start from the ORIGINAL labels).

Answers the meta-review: *do a blinded annotation audit across both inference modes*. This notebook pools the apparent false
positives (conf >= 0.25, no ground-truth box with IoU >= 0.1) of **standard** and **SAHI-320**, merges boxes that are the
same object, **shuffles** them and draws numbered crops with **no score and no mode** - the reviewer cannot tell which mode
produced a box.

After it finishes - **outputs**
* `audit_for_annotators.zip` -> give this to the annotators (sheets for A = all boxes, sheets for B = a random 25 %,
  two empty `verdicts_*.csv`). **Do not give them anything else from this notebook.**
* `results/audit/audit_key.csv` + `audit_meta.json` -> the PRIVATE key (which mode produced which box). Output tab -> New
  Dataset **`sat-audit-key`**, used only by 14B.

**Annotators:** open the PNG pages, and in `verdicts_A.csv` (B: `verdicts_B.csv`) change the `verdict` of an id from `N` to
`M` = a real plane / ship / vehicle that has **no label**, `P` = the red box covers only part of a real object,
`U` = cannot decide; leave `N` = nothing there / wrong. A and B work **independently**.

**Input setup** - right panel -> *Add Input* -> search the name -> *Add*. Links (your own Datasets are private: open them from the account that owns them, or share them with this account first):

* `sat-ext` - code (upload the sat_ext folder of the zip): https://www.kaggle.com/datasets/rifat4018/sat-ext
* `satelite-coco` - Roboflow COCO export, 640x640 (or download: universe.roboflow.com/rifat4018/aerial-object-detection-u3f5u): https://www.kaggle.com/datasets/rifat4018/satelite-coco
* `sat-splits` - output of notebook 01: https://www.kaggle.com/datasets/rifat4018/sat-splits
* `sat-results-e` - outputs of earlier notebooks (Output tab of that notebook -> *New Dataset*, same name); search them under *Your Datasets*.
* Check the printed input tree in the next cell before running the rest.

In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rollsroyce1/
      /kaggle/input/datasets/rollsroyce1/sat-ext/
        /kaggle/input/datasets/rollsroyce1/sat-ext/sat_ext/
          README.md
          analysis.py
          audit.py
          blindaudit.py
          config.py
          dataset.py
          ... +13 more files
      /kaggle/input/datasets/rollsroyce1/sat-results-e/
        yolo11n.pt
        yolo11s.pt
        yolo26n.pt
        /kaggle/input/datasets/rollsroyce1/sat-results-e/data_grouped/
          data.yaml
        /kaggle/input/datasets/rollsroyce1/sat-results-e/results/
        /kaggle/input/datasets/rollsroyce1/sat-results-e/results_E/
        /kaggle/input/datasets/rollsroyce1/sat-results-e/weights/
          grouped__YOLO11n__s42_best.pt
          grouped__YOLO11n__s43_best.pt
          grouped__YOLO11n__s44_best.pt
          grouped__YOLO11s__s42_best.pt
          grouped__YOLO11s__s43_best.pt
          grouped__YOLO11s__s44_best.pt
      /kag

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rollsroyce1/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
utils.warn_if_gpu_wasted()
import cv2, json, shutil, zipfile
import blindaudit as ba, dataset as ds, analysis, report

MODES = ba.AUDIT_MODES                 # ["standard", "sahi320"]
MAX_PER_MODE = None                    # None = review EVERY apparent false positive (fair to both modes). A number = RANDOM sample per mode.
FRAC_B = 0.25                          # share of the boxes the second annotator re-checks (for agreement)
RUN_PREF = ["grouped__YOLO11s__s42", "grouped__YOLO11s-1024__s42", "grouped__YOLO11m__s42"]

runs = {}
for p in utils.find_files("eval_test.json"):
    f = os.path.dirname(p)
    if all(utils.dets_path(f, m) for m in MODES):
        runs[utils.load_json(p)["run_id"]] = f
RID = next((r for r in RUN_PREF if r in runs), None)
assert RID, "no run with saved detections for %s - attach sat-results-e (grouped YOLO11s seed 42 with its results/ folder)" % MODES
dets = {m: analysis._dets(runs[RID], m) for m in MODES}
gt = analysis._gt_for("grouped")
assert "corrected" not in str(gt.get("info", "")), "corrected labels are attached - remove the notebook-13 outputs (sat-results-13, sat-test-relabeled)"
cands = ba.build_candidates(gt, dets, max_per_mode=MAX_PER_MODE)
print(RID, "| test images:", len(gt["images"]), "| GT boxes:", len(gt["annotations"]))
for m in MODES:
    print(f"  apparent false positives of {m}: {int(cands['in_' + m].sum())}")
print("distinct boxes to review:", len(cands), "| both modes:", int(cands[[f'in_{m}' for m in MODES]].all(axis=1).sum()),
      "| pages for A:", -(-len(cands) // 20), "| ~minutes for A at 8 s/page:", round(-(-len(cands) // 20) * 8 / 60))

grouped__YOLO11s__s42 | test images: 82 | GT boxes: 2735
  apparent false positives of standard: 402
  apparent false positives of sahi320: 925
distinct boxes to review: 931 | both modes: 396 | pages for A: 47 | ~minutes for A at 8 s/page: 6


In [4]:
info = ds.materialize("grouped")
img_dir = info["images"]["test"]
ids_b = ba.second_subset(cands, FRAC_B)
out = os.path.join(cfg.WORK_DIR, "audit_for_annotators")
shutil.rmtree(out, ignore_errors=True)
pa = ba.make_sheets(cands, img_dir, os.path.join(out, "sheets_A"))
pb = ba.make_sheets(cands, img_dir, os.path.join(out, "sheets_B"), ids=ids_b, order_seed=cfg.SEED + 2)   # other order
ba.write_templates(cands, out, ids_b)
kd = os.path.join(report.PAPER_DIR, "..", "audit")
os.makedirs(kd, exist_ok=True)
cands.to_csv(os.path.join(kd, "audit_key.csv"), index=False)
utils.save_json(dict(run_id=RID, modes=MODES, seed=cfg.SEED, ids_b=ids_b, n=len(cands), max_per_mode=MAX_PER_MODE), os.path.join(kd, "audit_meta.json"))
zp = os.path.join(cfg.WORK_DIR, "audit_for_annotators.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(out):
        for fn in files:
            z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), out))
shutil.rmtree(out, ignore_errors=True)
shutil.rmtree(info["root"], ignore_errors=True)
print("annotator pack:", zp, round(os.path.getsize(zp) / 1e6, 1), "MB | A pages:", len(pa), "| B pages:", len(pb), "| B boxes:", len(ids_b))
print("private key  :", os.path.join(kd, "audit_key.csv"), "(do NOT send to the annotators)")
try:
    from IPython.display import FileLink, display
    display(FileLink("audit_for_annotators.zip"))
except ImportError:
    pass

materialized 'grouped' (f7cee44b1365) -> /kaggle/working/data_grouped  images {'train': 984, 'val': 80, 'test': 82}
annotator pack: /kaggle/working/audit_for_annotators.zip 114.5 MB | A pages: 47 | B pages: 12 | B boxes: 233
private key  : /kaggle/working/results/paper/../audit/audit_key.csv (do NOT send to the annotators)


/kaggle/working/audit_for_annotators.zip